In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import re

df = pd.read_excel("Online-Retail.xlsx")

In [2]:
print("Kolon isimleri:")
print(df.columns.tolist())

Kolon isimleri:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [3]:
print("Satır sayısı:", df.shape[0])
print("Sütun sayısı:", df.shape[1])

Satır sayısı: 541909
Sütun sayısı: 8


In [4]:
bos_sutunlar = df.columns[df.isnull().all()]

print("Tamamen boş sütun sayısı:", len(bos_sutunlar))
print("Tamamen boş sütunlar:", bos_sutunlar.tolist())

Tamamen boş sütun sayısı: 0
Tamamen boş sütunlar: []


In [5]:
eksik_veriler = pd.DataFrame({
    "Eksik Değer Sayısı": df.isnull().sum(),
    "Eksik Değer Oranı (%)": (df.isnull().sum() / len(df)) * 100
})

print(eksik_veriler.sort_values(
    by="Eksik Değer Sayısı",
    ascending=False
))

             Eksik Değer Sayısı  Eksik Değer Oranı (%)
CustomerID               135080              24.926694
Description                1454               0.268311
StockCode                     0               0.000000
InvoiceNo                     0               0.000000
Quantity                      0               0.000000
InvoiceDate                   0               0.000000
UnitPrice                     0               0.000000
Country                       0               0.000000


In [6]:
print("CustomerID boş değer sayısı:", df["CustomerID"].isnull().sum())

print("CustomerID dolu değer sayısı:", df["CustomerID"].notnull().sum())

print("CustomerID eksik oranı (%):",
      df["CustomerID"].isnull().mean() * 100)

CustomerID boş değer sayısı: 135080
CustomerID dolu değer sayısı: 406829
CustomerID eksik oranı (%): 24.926694334288598


In [7]:
iptal_siparisler = df[
    df["InvoiceNo"].astype(str).str.startswith("C", na=False)
].copy()

print("\u0130ptal edilen i\u015flem sat\u0131r\u0131 say\u0131s\u0131:", len(iptal_siparisler))
print("\u0130ptal edilen fatura say\u0131s\u0131:", iptal_siparisler["InvoiceNo"].nunique())

iptal_siparisler.head()

İptal edilen işlem satırı sayısı:

 9288
İptal edilen fatura sayısı: 3836


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [8]:
print("Normal işlem sayısı:",
      (~df["InvoiceNo"].astype(str).str.startswith("C")).sum())

Normal işlem sayısı: 532621


In [9]:
iptal_siparisler["Tutar"] = (
    iptal_siparisler["Quantity"] * iptal_siparisler["UnitPrice"]
)

print("C ile ba\u015flayan iptal sat\u0131rlar\u0131n\u0131n net tutar\u0131:",
      iptal_siparisler["Tutar"].sum())

C ile başlayan iptal satırlarının net tutarı: -896812.49


In [10]:
# Ham df korunur; t\u00fcm t\u00fcket/iadeler net tutar kontrolleri i\u00e7in tutulur.
transaction_data = df.copy()
transaction_data["InvoiceDate"] = pd.to_datetime(transaction_data["InvoiceDate"])
transaction_data["LineTotal"] = (
    transaction_data["Quantity"] * transaction_data["UnitPrice"]
)
transaction_data["IsCancellation"] = (
    transaction_data["InvoiceNo"].astype(str).str.startswith("C", na=False)
)
transaction_data["TotalPrice"] = transaction_data["LineTotal"]

# Kimli\u011fi olmayan i\u015flemler genel ciro kontrollerinde kal\u0131r, m\u00fc\u015fteri RFM'ine girmez.
rfm_df = transaction_data.loc[transaction_data["CustomerID"].notna()].copy()
completed_sales_df = rfm_df.loc[
    (~rfm_df["IsCancellation"])
    & rfm_df["Quantity"].gt(0)
    & rfm_df["UnitPrice"].gt(0)
].copy()

# Negatif tutarlar Monetary'den d\u00fc\u015f\u00fcl\u00fcr; C ve C olmayan ayarlamalar ayr\u0131 raporlan\u0131r.
gross_revenue = transaction_data.loc[
    transaction_data["LineTotal"].gt(0), "LineTotal"
].sum()
negative_adjustments = transaction_data.loc[
    transaction_data["LineTotal"].lt(0), "LineTotal"
].sum()
net_revenue = transaction_data["LineTotal"].sum()
cancellation_total = transaction_data.loc[
    transaction_data["IsCancellation"], "LineTotal"
].sum()
other_negative_total = transaction_data.loc[
    (~transaction_data["IsCancellation"])
    & transaction_data["LineTotal"].lt(0), "LineTotal"
].sum()
missing_id_net = transaction_data.loc[
    transaction_data["CustomerID"].isna(), "LineTotal"
].sum()
identified_customer_net = rfm_df["LineTotal"].sum()

print("Ham sat\u0131r say\u0131s\u0131:", len(df))
print("CustomerID dolu sat\u0131r say\u0131s\u0131:", len(rfm_df))
print("CustomerID eksik sat\u0131r say\u0131s\u0131:", transaction_data["CustomerID"].isna().sum())
print("CustomerID eksik sat\u0131rlar\u0131n net tutar\u0131:", round(missing_id_net, 2))
print("RFM'e uygun tamamlanm\u0131\u015f sat\u0131\u015f sat\u0131r\u0131:", len(completed_sales_df))
print("C fatural\u0131 sat\u0131r / fatura say\u0131s\u0131:",
      int(transaction_data["IsCancellation"].sum()),
      transaction_data.loc[transaction_data["IsCancellation"], "InvoiceNo"].nunique())
print("C fatural\u0131 negatif tutar (iade/iptal etkisi):", round(cancellation_total, 2))
print("CustomerID eksik C sat\u0131r say\u0131s\u0131 / negatif tutar:",
      int((transaction_data["IsCancellation"] & transaction_data["CustomerID"].isna()).sum()),
      round(transaction_data.loc[transaction_data["IsCancellation"] & transaction_data["CustomerID"].isna(), "LineTotal"].sum(), 2))
print("C olmayan negatif tutar ayarlamalar\u0131:", round(other_negative_total, 2))
print("Negatif Quantity sat\u0131rlar\u0131 (C olmayan):",
      int(((~transaction_data["IsCancellation"]) & transaction_data["Quantity"].lt(0)).sum()))
print("S\u0131f\u0131r / negatif sat\u0131r tutar\u0131 say\u0131s\u0131:",
      int(transaction_data["LineTotal"].eq(0).sum()),
      int(transaction_data["LineTotal"].lt(0).sum()))
print("S\u0131f\u0131r / negatif UnitPrice sat\u0131r\u0131 say\u0131s\u0131:",
      int(transaction_data["UnitPrice"].eq(0).sum()),
      int(transaction_data["UnitPrice"].lt(0).sum()))
print("Mutlak Quantity > 100 / > 1000 sat\u0131r say\u0131s\u0131 (silinmedi):",
      int(transaction_data["Quantity"].abs().gt(100).sum()),
      int(transaction_data["Quantity"].abs().gt(1000).sum()))
print("En y\u00fcksek Quantity:", transaction_data["Quantity"].max())
print("En y\u00fcksek miktarl\u0131 ilk 5 sat\u0131r:")
print(transaction_data.nlargest(5, "Quantity")[[
    "InvoiceNo", "StockCode", "Quantity", "UnitPrice", "LineTotal", "CustomerID"
]].to_string(index=False))

# Financial controls: positive line amounts + negative adjustments = signed net.
assert np.isclose(gross_revenue + negative_adjustments, net_revenue)
assert np.isclose(identified_customer_net + missing_id_net, net_revenue)

# Periodic and main RFM analyses use this same transaction-level rule set.
def rfm_hesapla(veri, analiz_tarihi):
    calisma = veri.loc[veri["CustomerID"].notna()].copy()
    calisma["InvoiceDate"] = pd.to_datetime(calisma["InvoiceDate"])
    calisma["LineTotal"] = calisma["Quantity"] * calisma["UnitPrice"]
    iptal = calisma["InvoiceNo"].astype(str).str.startswith("C", na=False)
    tamamlanan_satis = calisma.loc[
        (~iptal) & calisma["Quantity"].gt(0) & calisma["UnitPrice"].gt(0)
    ]
    if tamamlanan_satis.empty:
        raise ValueError("RFM i\u00e7in tamamlanm\u0131\u015f sat\u0131\u015f i\u015flemi bulunamad\u0131.")

    analiz_tarihi = pd.Timestamp(analiz_tarihi)
    if (tamamlanan_satis["InvoiceDate"] > analiz_tarihi).any():
        raise ValueError("Analiz tarihi, d\u00f6nemdeki sat\u0131\u015f tarihinden \u00f6nce olamaz.")

    sonuc = tamamlanan_satis.groupby("CustomerID").agg(
        Recency=("InvoiceDate", lambda tarihler: (analiz_tarihi - tarihler.max()).days),
        Frequency=("InvoiceNo", "nunique")
    )
    net_tutar = calisma.groupby("CustomerID")["LineTotal"].sum().rename("Monetary")
    sonuc = sonuc.join(net_tutar, how="left")

    # Mevcut i\u015f kural\u0131 korunur: net Monetary <= 0 olanlar segment havuzuna girmez.
    return sonuc.loc[sonuc["Monetary"].gt(0)].copy()

Ham satır sayısı: 541909
CustomerID dolu satır sayısı: 406829
CustomerID eksik satır sayısı: 135080
CustomerID eksik satırların net tutarı: 1447682.12
RFM'e uygun tamamlanmış satış satırı: 397884
C faturalı satır / fatura sayısı: 9288 3836
C faturalı negatif tutar (iade/iptal etkisi): -896812.49
CustomerID eksik C satır sayısı / negatif tutar: 383 -285470.4
C olmayan negatif tutar ayarlamaları: -22124.12
Negatif Quantity satırları (C olmayan): 1336
Sıfır / negatif satır tutarı sayısı: 2515 9290
Sıfır / negatif UnitPrice satırı sayısı: 2515 2
Mutlak Quantity > 100 / > 1000 satır sayısı (silinmedi): 5406 174
En yüksek Quantity: 80995
En yüksek miktarlı ilk 5 satır:
InvoiceNo StockCode  Quantity  UnitPrice  LineTotal  CustomerID
   581483     23843     80995       2.08   168469.6     16446.0
   541431     23166     74215       1.04    77183.6     12346.0
   578841     84826     12540       0.00        0.0     13256.0
   542504     37413      5568       0.00        0.0         NaN
   57300

In [11]:
# Kaydedilmi\u015f eski y\u00f6ntem: C faturalar\u0131 R/F'yi etkiliyor, t\u00fcm hareketler M'ye net ekleniyor.
legacy_reference_date = rfm_df["InvoiceDate"].max() + dt.timedelta(days=1)
legacy_rfm = rfm_df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda dates: (legacy_reference_date - dates.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("LineTotal", "sum")
)
legacy_rfm = legacy_rfm.loc[legacy_rfm["Monetary"].gt(0)].copy()

# Yeni R/F sadece C olmayan, pozitif miktar ve fiyatl\u0131 tamamlanm\u0131\u015f faturalardan;
# Monetary ise m\u00fc\u015fterinin t\u00fcm bilinen pozitif/negatif hareketlerinin net toplam\u0131d\u0131r.
today_date = completed_sales_df["InvoiceDate"].max() + dt.timedelta(days=1)
rfm = rfm_hesapla(rfm_df, today_date)

purchase_customers = completed_sales_df["CustomerID"].nunique()
customer_net = rfm_df.groupby("CustomerID")["LineTotal"].sum()
buyer_net = customer_net.reindex(completed_sales_df["CustomerID"].drop_duplicates())
nonpositive_net_buyers = int(buyer_net.le(0).sum())
zero_net_buyers = int(buyer_net.eq(0).sum())
negative_net_buyers = int(buyer_net.lt(0).sum())
no_sale_customer_net = customer_net.loc[~customer_net.index.isin(buyer_net.index)]
excluded_rfm_net = customer_net.loc[~customer_net.index.isin(rfm.index)].sum()

print("Eski kuralla RFM m\u00fc\u015fteri say\u0131s\u0131:", len(legacy_rfm))
print("Yeni kuralla RFM m\u00fc\u015fteri say\u0131s\u0131:", len(rfm))
print("Tamamlanm\u0131\u015f sat\u0131\u015f yapan benzersiz m\u00fc\u015fteri:", purchase_customers)
print("Net Monetary <= 0 oldu\u011fu i\u00e7in RFM d\u0131\u015f\u0131nda kalan al\u0131c\u0131:", nonpositive_net_buyers)
print("Bu al\u0131c\u0131lar\u0131n s\u0131f\u0131r / negatif net say\u0131s\u0131:", zero_net_buyers, negative_net_buyers)
print("Net pozitif sat\u0131\u015f\u0131 olmayan m\u00fc\u015fteri say\u0131s\u0131 / net tutar\u0131:",
      len(no_sale_customer_net), round(no_sale_customer_net.sum(), 2))
print("RFM d\u0131\u015f\u0131nda kalan tan\u0131ml\u0131 m\u00fc\u015fterilerin net tutar\u0131:",
      round(excluded_rfm_net, 2))

Eski kuralla RFM müşteri sayısı: 4322
Yeni kuralla RFM müşteri sayısı: 4322
Tamamlanmış satış yapan benzersiz müşteri: 4338
Net Monetary <= 0 olduğu için RFM dışında kalan alıcı: 16
Bu alıcıların sıfır / negatif net sayısı: 7 9
Net pozitif satışı olmayan müşteri sayısı / net tutarı: 34 -10411.24
RFM dışında kalan tanımlı müşterilerin net tutarı: -13229.14


In [12]:
print("Yeni RFM boyutu:", rfm.shape)
print(rfm.head())

karsilastirma = legacy_rfm[["Recency", "Frequency", "Monetary"]].join(
    rfm[["Recency", "Frequency", "Monetary"]],
    how="inner",
    lsuffix="_Eski",
    rsuffix="_Yeni"
)
print("Eski/yeni ortak m\u00fc\u015fteri say\u0131s\u0131:", len(karsilastirma))
print("Recency de\u011fi\u015fen m\u00fc\u015fteri:",
      int(karsilastirma["Recency_Eski"].ne(karsilastirma["Recency_Yeni"]).sum()))
print("Frequency de\u011fi\u015fen m\u00fc\u015fteri:",
      int(karsilastirma["Frequency_Eski"].ne(karsilastirma["Frequency_Yeni"]).sum()))
print("Ortak m\u00fc\u015fterilerde Monetary net tutar\u0131 korundu mu?:",
      bool(np.allclose(karsilastirma["Monetary_Eski"], karsilastirma["Monetary_Yeni"])))
assert np.allclose(karsilastirma["Monetary_Eski"], karsilastirma["Monetary_Yeni"])

Yeni RFM boyutu: (4322, 3)
            Recency  Frequency  Monetary
CustomerID                              
12347.0           2          7   4310.00
12348.0          75          4   1797.24
12349.0          19          1   1757.55
12350.0         310          1    334.40
12352.0          36          8   1545.41
Eski/yeni ortak müşteri sayısı: 4322
Recency değişen müşteri: 437
Frequency değişen müşteri: 1541
Ortak müşterilerde Monetary net tutarı korundu mu?: True


In [13]:
# Frequency scoring: equal invoice counts receive identical scores; bucket sizes may differ.
def frequency_score(series, score_count=5):
    """Use nearest feasible empirical quintile cutoffs without splitting tied counts."""
    if series.isna().any():
        raise ValueError("Frequency scores require non-null invoice counts.")
    counts = series.value_counts().sort_index()
    values = counts.index.to_numpy()
    if len(values) < score_count:
        raise ValueError("Not enough distinct Frequency values to create all score groups.")

    cumulative_share = counts.cumsum().to_numpy() / counts.sum()
    cutoffs = []
    previous_index = -1
    for quantile_number in range(1, score_count):
        first_index = previous_index + 1
        last_index = len(values) - (score_count - quantile_number) - 1
        candidates = range(first_index, last_index + 1)
        cutoff_index = min(
            candidates,
            key=lambda index: abs(cumulative_share[index] - quantile_number / score_count)
        )
        cutoffs.append(values[cutoff_index])
        previous_index = cutoff_index

    scores = np.searchsorted(np.asarray(cutoffs), series.to_numpy(), side="left") + 1
    return pd.Series(scores, index=series.index, dtype=int)

# Keep existing R/F segment logic; Monetary remains in full RFM code and reporting.
rfm["R_Score"] = pd.qcut(rfm["Recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_Score"] = frequency_score(rfm["Frequency"])
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM_Code"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

print("F_Score method: nearest feasible quintile cutoffs; tied Frequency values stay together.")
print("F_Score customers by score (unequal group sizes are expected):")
print(rfm["F_Score"].value_counts().sort_index())
print("Same Frequency value always maps to one F score:",
      rfm.groupby("Frequency")["F_Score"].nunique().max() == 1)
assert rfm.groupby("Frequency")["F_Score"].nunique().max() == 1
assert set(rfm["F_Score"].unique()) == set(range(1, 6))
print(rfm.head())

F_Score method: nearest feasible quintile cutoffs; tied Frequency values stay together.
F_Score customers by score (unequal group sizes are expected):
F_Score
1    1480
2     832
3     508
4     630
5     872
Name: count, dtype: int64
Same Frequency value always maps to one F score: True
            Recency  Frequency  Monetary  R_Score  F_Score  M_Score RFM_Code
CustomerID                                                                  
12347.0           2          7   4310.00        5        5        5      555
12348.0          75          4   1797.24        2        4        4      244
12349.0          19          1   1757.55        4        1        4      414
12350.0         310          1    334.40        1        1        2      112
12352.0          36          8   1545.41        3        5        4      354


In [14]:
# Var olan segment isimleri ve R-F i\u015f mant\u0131\u011f\u0131 korunur; M skorunu ayr\u0131ca raporlar\u0131z.
seg_map = {
    r'[1-2][1-2]': 'Uyuyanlar (Hibernating)',
    r'[1-2][3-4]': 'Risk Alt\u0131ndakiler (At Risk)',
    r'[1-2]5': 'Kaybedilemezler (Cant Lose)',
    r'3[1-2]': 'Uyumak \u00dczere Olanlar (About To Sleep)',
    r'33': 'Dikkat Gerektirenler (Need Attention)',
    r'[3-4][4-5]': 'Sad\u0131k M\u00fc\u015fteriler (Loyal Customers)',
    r'41': 'Umut Verenler (Promising)',
    r'51': 'Yeni M\u00fc\u015fteriler (New Customers)',
    r'[4-5][2-3]': 'Potansiyel Sad\u0131klar (Potential Loyalists)',
    r'5[4-5]': '\u015eampiyonlar (Champions)'
}

def segment_belirle(r, f):
    skor = f"{int(r)}{int(f)}"
    for desen, segment in seg_map.items():
        if re.fullmatch(desen, skor):
            return segment
    return "Tan\u0131ms\u0131z"

legacy_rfm["R_Score"] = pd.qcut(
    legacy_rfm["Recency"], 5, labels=[5, 4, 3, 2, 1]
)
legacy_rfm["F_Score"] = pd.qcut(
    legacy_rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]
)

# Prior balanced quintile method, recomputed on the same customers for a direct comparison.
rfm["F_Score_Balanced"] = pd.qcut(
    rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]
).astype(int)
rfm["Segment_Balanced"] = (
    rfm["R_Score"].astype(str) + rfm["F_Score_Balanced"].astype(str)
).replace(seg_map, regex=True)

for tablo in (legacy_rfm, rfm):
    tablo["Segment"] = tablo.apply(
        lambda satir: segment_belirle(satir["R_Score"], satir["F_Score"]),
        axis=1
    )

assert rfm["Segment"].ne("Tan\u0131ms\u0131z").all()
assert legacy_rfm["Segment"].ne("Tan\u0131ms\u0131z").all()
segment_karsilastirma = pd.concat(
    [legacy_rfm["Segment"].value_counts().rename("Eski"),
     rfm["Segment"].value_counts().rename("Yeni")],
    axis=1
).fillna(0).astype(int)
segment_karsilastirma["Fark"] = (
    segment_karsilastirma["Yeni"] - segment_karsilastirma["Eski"]
)
print("Eski / yeni segment m\u00fc\u015fteri say\u0131lar\u0131:")
print(segment_karsilastirma.sort_values("Yeni", ascending=False))
print("Toplam:", int(segment_karsilastirma["Eski"].sum()),
      int(segment_karsilastirma["Yeni"].sum()))

f_skor_karsilastirma = pd.concat(
    [rfm["F_Score_Balanced"].value_counts().rename("Eski_Dengeli"),
     rfm["F_Score"].value_counts().rename("Yeni_Esitlik_Korumali")],
    axis=1
).fillna(0).astype(int).sort_index()
f_skor_karsilastirma["Fark"] = (
    f_skor_karsilastirma["Yeni_Esitlik_Korumali"] - f_skor_karsilastirma["Eski_Dengeli"]
)
print("\nF score counts: old balanced quintiles vs new tie-preserving method:")
print(f_skor_karsilastirma)

segment_f_karsilastirma = pd.concat(
    [rfm["Segment_Balanced"].value_counts().rename("Eski_Dengeli"),
     rfm["Segment"].value_counts().rename("Yeni_Esitlik_Korumali")],
    axis=1
).fillna(0).astype(int)
segment_f_karsilastirma["Fark"] = (
    segment_f_karsilastirma["Yeni_Esitlik_Korumali"] - segment_f_karsilastirma["Eski_Dengeli"]
)
print("\nSegment customer counts on the same RFM population: old F vs new F:")
print(segment_f_karsilastirma.sort_values("Yeni_Esitlik_Korumali", ascending=False))
assert rfm.groupby("Frequency")["F_Score"].nunique().max() == 1
assert int(segment_f_karsilastirma["Eski_Dengeli"].sum()) == len(rfm)
assert int(segment_f_karsilastirma["Yeni_Esitlik_Korumali"].sum()) == len(rfm)

Eski / yeni segment müşteri sayıları:
                                           Eski  Yeni  Fark
Segment                                                    
Uyuyanlar (Hibernating)                    1041  1328   287
Sadık Müşteriler (Loyal Customers)          794   686  -108
Şampiyonlar (Champions)                     640   604   -36
Potansiyel Sadıklar (Potential Loyalists)   486   480    -6
Uyumak Üzere Olanlar (About To Sleep)       365   476   111
Risk Altındakiler (At Risk)                 610   332  -278
Umut Verenler (Promising)                    95   165    70
Dikkat Gerektirenler (Need Attention)       180   118   -62
Yeni Müşteriler (New Customers)              41    67    26
Kaybedilemezler (Cant Lose)                  70    66    -4
Toplam: 4322 4322

F score counts: old balanced quintiles vs new tie-preserving method:
   Eski_Dengeli  Yeni_Esitlik_Korumali  Fark
1           865                   1480   615
2           864                    832   -32
3           864    

In [15]:
segment_summary = rfm.groupby("Segment", observed=True).agg(
    Ort_Recency=("Recency", "mean"),
    Ort_Frequency=("Frequency", "mean"),
    Ort_Monetary=("Monetary", "mean"),
    Toplam_Ciro=("Monetary", "sum"),
    Musteri_Sayisi=("Monetary", "size"),
    Ort_M_Skor=("M_Score", lambda skor: skor.astype(int).mean())
).sort_values("Toplam_Ciro", ascending=False)

# Bu pay, yaln\u0131zca pozitif net Monetary'li RFM m\u00fc\u015fterileri i\u00e7indeki payd\u0131r.
rfm_net_total = segment_summary["Toplam_Ciro"].sum()
segment_summary["Ciro_Payi_%"] = (
    segment_summary["Toplam_Ciro"] / rfm_net_total * 100
).round(2)

assert int(segment_summary["Musteri_Sayisi"].sum()) == len(rfm)
assert np.isclose(segment_summary["Ciro_Payi_%"].sum(), 100, atol=0.05)
assert np.isclose(rfm_net_total, rfm["Monetary"].sum())
assert np.isclose(gross_revenue + negative_adjustments, net_revenue)
assert np.isclose(identified_customer_net + missing_id_net, net_revenue)

print("Segment da\u011f\u0131l\u0131m\u0131 ve RFM i\u00e7i net ciro pay\u0131:")
print(segment_summary[["Musteri_Sayisi", "Toplam_Ciro", "Ciro_Payi_%", "Ort_M_Skor"]])
print("RFM segment net Monetary:", round(rfm_net_total, 2))
print("T\u00fcm veri net gelir:", round(net_revenue, 2))
print("RFM segment netinin toplam net gelire oran\u0131 (%):",
      round(rfm_net_total / net_revenue * 100, 2))
print("Ciro kontrolleri | br\u00fct pozitif tutar:", round(gross_revenue, 2),
      "| negatif ayarlamalar:", round(negative_adjustments, 2),
      "| net:", round(net_revenue, 2),
      "| kontrol:", bool(np.isclose(gross_revenue + negative_adjustments, net_revenue)))
print("C faturalar\u0131 negatif tutar:", round(cancellation_total, 2),
      "| C olmayan negatif sat\u0131r tutar\u0131:", round(other_negative_total, 2))
print("CustomerID dolu net tutar + eksik ID net tutar:",
      round(identified_customer_net, 2), "+", round(missing_id_net, 2),
      "=", round(identified_customer_net + missing_id_net, 2))
print("Segment x M_Score da\u011f\u0131l\u0131m\u0131 (%):")
print((pd.crosstab(rfm["Segment"], rfm["M_Score"], normalize="index") * 100).round(1))
balanced_segment_summary = rfm.groupby("Segment_Balanced").agg(
    Musteri_Sayisi=("Monetary", "size"),
    Toplam_Ciro=("Monetary", "sum")
)
balanced_segment_summary["Ciro_Payi_%"] = (
    balanced_segment_summary["Toplam_Ciro"] / rfm["Monetary"].sum() * 100
)
new_segment_comparison = segment_summary[["Musteri_Sayisi", "Toplam_Ciro"]].copy()
new_segment_comparison["Ciro_Payi_Yeni_%"] = (
    new_segment_comparison["Toplam_Ciro"] / rfm["Monetary"].sum() * 100
)
score_method_comparison = balanced_segment_summary[
    ["Musteri_Sayisi", "Ciro_Payi_%"]
].rename(columns={
    "Musteri_Sayisi": "Musteri_Eski_Dengeli",
    "Ciro_Payi_%": "Ciro_Payi_Eski_%"
}).join(
    new_segment_comparison[["Musteri_Sayisi", "Ciro_Payi_Yeni_%"]].rename(columns={
        "Musteri_Sayisi": "Musteri_Yeni_Esitlik_Korumali"
    }),
    how="outer"
).fillna(0)
score_method_comparison["Musteri_Fark"] = (
    score_method_comparison["Musteri_Yeni_Esitlik_Korumali"]
    - score_method_comparison["Musteri_Eski_Dengeli"]
)
score_method_comparison["Ciro_Payi_Farki_pp"] = (
    score_method_comparison["Ciro_Payi_Yeni_%"]
    - score_method_comparison["Ciro_Payi_Eski_%"]
)
assert np.isclose(score_method_comparison["Ciro_Payi_Eski_%"].sum(), 100)
assert np.isclose(score_method_comparison["Ciro_Payi_Yeni_%"].sum(), 100)
print("\nSegment-level old/new F scoring comparison:")
print(score_method_comparison.sort_values(
    "Ciro_Payi_Yeni_%", ascending=False
).round(2))
print("Segment revenue share totals, old/new (%):",
      round(score_method_comparison["Ciro_Payi_Eski_%"].sum(), 2),
      round(score_method_comparison["Ciro_Payi_Yeni_%"].sum(), 2))


Segment dağılımı ve RFM içi net ciro payı:
                                           Musteri_Sayisi  Toplam_Ciro  \
Segment                                                                  
Şampiyonlar (Champions)                               604  4188278.560   
Sadık Müşteriler (Loyal Customers)                    686  2090845.991   
Uyuyanlar (Hibernating)                              1328   591597.322   
Risk Altındakiler (At Risk)                           332   411755.921   
Potansiyel Sadıklar (Potential Loyalists)             480   405475.340   
Uyumak Üzere Olanlar (About To Sleep)                 476   243229.870   
Kaybedilemezler (Cant Lose)                            66   179591.100   
Dikkat Gerektirenler (Need Attention)                 118   128083.200   
Umut Verenler (Promising)                             165    51411.540   
Yeni Müşteriler (New Customers)                        67    23026.110   

                                           Ciro_Payi_%  Ort_M_Skor  

In [16]:
risk_segment = "Risk Alt\u0131ndakiler (At Risk)"
risk_musteri = int(segment_summary.loc[risk_segment, "Musteri_Sayisi"])
risk_ciro = float(segment_summary.loc[risk_segment, "Toplam_Ciro"])

# %10, veri kaynakl\u0131 tahmin de\u011fil; yaln\u0131zca senaryo varsay\u0131m\u0131d\u0131r.
geri_kazanim_orani = 0.10
geri_kazanilan_musteri = round(risk_musteri * geri_kazanim_orani)
ortalama_ciro = risk_ciro / risk_musteri
tahmini_ek_gelir = geri_kazanilan_musteri * ortalama_ciro

print("Senaryo varsay\u0131m\u0131 - geri kazan\u0131lacak m\u00fc\u015fteri:", geri_kazanilan_musteri)
print("Risk segmenti m\u00fc\u015fteri ba\u015f\u0131 net Monetary:", round(ortalama_ciro, 2))
print("Senaryo tahmini ek gelir (garanti edilen gelir de\u011fil):",
      round(tahmini_ek_gelir, 2))

Senaryo varsayımı - geri kazanılacak müşteri: 33
Risk segmenti müşteri başı net Monetary: 1240.23
Senaryo tahmini ek gelir (garanti edilen gelir değil): 40927.55


In [17]:
# 1. RFM tablosundaki Segment bilgisini, orijinal işlem verisiyle birleştiriyoruz
rfm_segments = rfm[['Segment']].reset_index()
merged_data = pd.merge(rfm_df, rfm_segments, on='CustomerID', how='inner')

# 2. Şampiyonlar en çok hangi ürünleri (Description) almış?
champions_top = merged_data[merged_data['Segment'] == 'Şampiyonlar (Champions)']
print("👑 ŞAMPİYONLARIN EN ÇOK ALDIĞI 5 ÜRÜN:")
print(champions_top.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(5))

print("\n" + "="*40 + "\n")

# 3. Uyuyanlar geçmişte en çok hangi ürünleri almış?
sleepers_top = merged_data[merged_data['Segment'] == 'Uyuyanlar (Hibernating)']
print("💤 UYUYANLARIN (TERK EDENLER) EN ÇOK ALDIĞI 5 ÜRÜN:")
print(sleepers_top.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(5))

👑 ŞAMPİYONLARIN EN ÇOK ALDIĞI 5 ÜRÜN:
Description
WORLD WAR 2 GLIDERS ASSTD DESIGNS     30215
JUMBO BAG RED RETROSPOT               25672
POPCORN HOLDER                        20600
WHITE HANGING HEART T-LIGHT HOLDER    18166
PACK OF 72 RETROSPOT CAKE CASES       15662
Name: Quantity, dtype: int64


💤 UYUYANLARIN (TERK EDENLER) EN ÇOK ALDIĞI 5 ÜRÜN:
Description
WORLD WAR 2 GLIDERS ASSTD DESIGNS    5472
SMALL POPCORN HOLDER                 4737
EMPIRE DESIGN ROSETTE                3936
PACK OF 72 RETROSPOT CAKE CASES      2881
ASSORTED COLOURS SILK FAN            2816
Name: Quantity, dtype: int64


In [18]:
# Net country revenue includes negative adjustments; order/customer counts use valid sales only.
country_net = rfm_df.groupby("Country")["TotalPrice"].sum().rename("Toplam_Ciro")
country_customers = completed_sales_df.groupby("Country")["CustomerID"].nunique().rename("Musteri_Sayisi")
country_orders = completed_sales_df.groupby("Country")["InvoiceNo"].nunique().rename("Siparis_Sayisi")
ulke_analizi = pd.concat(
    [country_customers, country_net, country_orders], axis=1
).fillna({"Musteri_Sayisi": 0, "Toplam_Ciro": 0, "Siparis_Sayisi": 0})
ulke_analizi["Musteri_Sayisi"] = ulke_analizi["Musteri_Sayisi"].astype(int)
ulke_analizi["Siparis_Sayisi"] = ulke_analizi["Siparis_Sayisi"].astype(int)
ulke_analizi["Ortalama_Siparis"] = (
    ulke_analizi["Toplam_Ciro"] /
    ulke_analizi["Siparis_Sayisi"].where(ulke_analizi["Siparis_Sayisi"].gt(0))
)
ulke_analizi = ulke_analizi.sort_values("Toplam_Ciro", ascending=False)
ulke_analizi["Ciro_Payi"] = (
    ulke_analizi["Toplam_Ciro"] / ulke_analizi["Toplam_Ciro"].sum() * 100
)
print("Net ciro t\u00fcm bilinen m\u00fc\u015fteri hareketlerinden; fatura say\u0131s\u0131 pozitif sat\u0131\u015flardan hesaplan\u0131r.")
print(ulke_analizi.round(2))

Net ciro tüm bilinen müşteri hareketlerinden; fatura sayısı pozitif satışlardan hesaplanır.
                      Musteri_Sayisi  Toplam_Ciro  Siparis_Sayisi  \
Country                                                             
United Kingdom                  3920   6767873.39           16646   
Netherlands                        9    284661.54              94   
EIRE                               3    250285.22             260   
Germany                           94    221698.21             457   
France                            87    196712.84             389   
Australia                          9    137077.27              57   
Switzerland                       21     55739.40              51   
Spain                             30     54774.58              90   
Belgium                           25     40910.96              98   
Sweden                             8     36595.91              36   
Japan                              8     35340.62              19   
Norway     

In [19]:
ulke_analizi["Musteri_Basina_Ciro"] = (
    ulke_analizi["Toplam_Ciro"] /
    ulke_analizi["Musteri_Sayisi"].where(ulke_analizi["Musteri_Sayisi"].gt(0))
)

satis_yapmayan_ulkeler = ulke_analizi.loc[
    ulke_analizi["Musteri_Sayisi"].eq(0),
    ["Toplam_Ciro", "Siparis_Sayisi"]
]
print("Tamamlanm\u0131\u015f sat\u0131\u015f m\u00fc\u015fterisi olmayan \u00fclkeler:")
print(satis_yapmayan_ulkeler)
print("M\u00fc\u015fteri ba\u015f\u0131 net ciroda ge\u00e7erli payda bulunmayanlar hari\u00e7 ilk 10:")
print(
    ulke_analizi[["Musteri_Sayisi", "Toplam_Ciro", "Musteri_Basina_Ciro"]]
    .dropna(subset=["Musteri_Basina_Ciro"])
    .sort_values("Musteri_Basina_Ciro", ascending=False)
    .head(10)
    .round(2)
)

Tamamlanmış satış müşterisi olmayan ülkeler:
Empty DataFrame
Columns: [Toplam_Ciro, Siparis_Sayisi]
Index: []
Müşteri başı net ciroda geçerli payda bulunmayanlar hariç ilk 10:
             Musteri_Sayisi  Toplam_Ciro  Musteri_Basina_Ciro
Country                                                      
EIRE                      3    250285.22             83428.41
Netherlands               9    284661.54             31629.06
Australia                 9    137077.27             15230.81
Singapore                 1      9120.39              9120.39
Sweden                    8     36595.91              4574.49
Japan                     8     35340.62              4417.58
Iceland                   1      4310.00              4310.00
Norway                   10     35163.46              3516.35
Switzerland              21     55739.40              2654.26
Germany                  94    221698.21              2358.49


In [20]:
# Net tutar t\u00fcm i\u015flemlerden, fatura frekans\u0131 tamamlanm\u0131\u015f sat\u0131\u015flardan hesaplan\u0131r.
musteri_ulke_net = rfm_df.groupby(
    ["Country", "CustomerID"]
)["TotalPrice"].sum().rename("Toplam_Ciro")
musteri_ulke_siparis = completed_sales_df.groupby(
    ["Country", "CustomerID"]
)["InvoiceNo"].nunique().rename("Siparis_Sayisi")
musteri_ulke = pd.concat(
    [musteri_ulke_net, musteri_ulke_siparis], axis=1
).fillna({"Siparis_Sayisi": 0}).reset_index()
musteri_ulke["Siparis_Sayisi"] = musteri_ulke["Siparis_Sayisi"].astype(int)

print(
    musteri_ulke[
        musteri_ulke["Country"].isin(["EIRE", "Netherlands", "Australia"])
    ].sort_values("Toplam_Ciro", ascending=False).to_string(index=False)
)

    Country  CustomerID  Toplam_Ciro  Siparis_Sayisi
Netherlands     14646.0    279489.02              73
       EIRE     14911.0    132572.62             201
  Australia     12415.0    123725.45              21
       EIRE     14156.0    113384.14              55
  Australia     12431.0      5443.61              13
       EIRE     14016.0      4328.46               4
  Australia     12388.0      2780.66               6
  Australia     12424.0      1760.96               2
  Australia     12393.0      1582.60               4
Netherlands     12759.0      1531.44               3
Netherlands     12775.0      1351.24               5
Netherlands     12778.0       804.23               6
  Australia     12434.0       765.19               2
Netherlands     12787.0       463.66               3
Netherlands     12802.0       412.58               1
  Australia     12386.0       401.90               2
  Australia     12422.0       386.20               2
Netherlands     12790.0       324.92          

In [21]:
rfm.reset_index().merge(
    musteri_ulke,
    on="CustomerID",
    how="inner"
).query(
    "Country in ['EIRE', 'Netherlands', 'Australia']"
).sort_values(
    "Toplam_Ciro", ascending=False
)[[
    "Country", "CustomerID", "Segment",
    "Recency", "Frequency", "Monetary",
    "Toplam_Ciro"
]].head(10)

,Country,CustomerID,Segment,Recency,Frequency,Monetary,Toplam_Ciro
1692,Netherlands,14646.0,Şampiyonlar (Champions),2,73,279489.02,279489.02
1881,EIRE,14911.0,Şampiyonlar (Champions),1,201,132572.62,132572.62
56,Australia,12415.0,Sadık Müşteriler (Loyal Customers),24,21,123725.45,123725.45
1338,EIRE,14156.0,Şampiyonlar (Champions),10,55,113384.14,113384.14
73,Australia,12431.0,Sadık Müşteriler (Loyal Customers),36,15,6416.39,5443.61
1235,EIRE,14016.0,Risk Altındakiler (At Risk),162,4,4328.46,4328.46
34,Australia,12388.0,Sadık Müşteriler (Loyal Customers),16,6,2780.66,2780.66
65,Australia,12424.0,Uyuyanlar (Hibernating),163,2,1760.96,1760.96
37,Australia,12393.0,Risk Altındakiler (At Risk),72,4,1582.60,1582.60
343,Netherlands,12759.0,Dikkat Gerektirenler (Need Attention),52,3,1531.44,1531.44


In [22]:
# D\u00f6nemler ayn\u0131 temizlenmi\u015f transaction tablosundan; ortak RFM fonksiyonu sat\u0131\u015f ve iade kurallar\u0131n\u0131 uygular.
donem1 = rfm_df.loc[
    rfm_df["InvoiceDate"] < pd.Timestamp("2011-06-01")
].copy()
donem2 = rfm_df.loc[
    rfm_df["InvoiceDate"] >= pd.Timestamp("2011-06-01")
].copy()

print("1. d\u00f6nem i\u015flem sat\u0131r\u0131:", len(donem1), "| 2. d\u00f6nem:", len(donem2))

1. dönem işlem satırı: 149053 | 2. dönem: 257776


In [23]:
# Ana RFM fonksiyonu cell 9'da tan\u0131ml\u0131; d\u00f6nemsel hesaplama ayn\u0131 kurallar\u0131 kullan\u0131r.
assert callable(rfm_hesapla)
print("Ana ve d\u00f6nemsel RFM ayn\u0131 hesaplama fonksiyonunu kullan\u0131yor.")

Ana ve dönemsel RFM aynı hesaplama fonksiyonunu kullanıyor.


In [24]:
# Eski d\u00f6nem y\u00f6ntemi yaln\u0131zca kar\u015f\u0131la\u015ft\u0131rma referans\u0131d\u0131r: iptal/iadeleri R ve F'ye katard\u0131.
def eski_donem_rfm(veri, analiz_tarihi):
    return veri.groupby("CustomerID").agg(
        Recency=("InvoiceDate", lambda tarihler: (analiz_tarihi - tarihler.max()).days),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("TotalPrice", "sum")
    )

legacy_rfm1 = eski_donem_rfm(donem1, pd.Timestamp("2011-06-01"))
legacy_rfm2 = eski_donem_rfm(donem2, pd.Timestamp("2012-01-01"))
rfm1 = rfm_hesapla(donem1, pd.Timestamp("2011-06-01"))
rfm2 = rfm_hesapla(donem2, pd.Timestamp("2012-01-01"))

print("Eski / yeni d\u00f6nem m\u00fc\u015fteri say\u0131lar\u0131:",
      (len(legacy_rfm1), len(rfm1)), (len(legacy_rfm2), len(rfm2)))

Eski / yeni dönem müşteri sayıları: (2767, 2708) (3577, 3532)


In [25]:
def skor_hesapla(veri, monetary_skoru=True):
    veri = veri.copy()
    veri["R_Score"] = pd.qcut(
        veri["Recency"].rank(method="first"),
        5,
        labels=[5, 4, 3, 2, 1]
    ).astype(int)
    veri["F_Score"] = frequency_score(veri["Frequency"])
    if monetary_skoru and "Monetary" in veri.columns:
        veri["M_Score"] = pd.qcut(
            veri["Monetary"], 5, labels=[1, 2, 3, 4, 5]
        ).astype(int)
    return veri

In [26]:
rfm1 = skor_hesapla(rfm1)
rfm2 = skor_hesapla(rfm2)

# Historical reference: the prior balanced, tie-splitting F scoring.
for legacy_period in (legacy_rfm1, legacy_rfm2):
    legacy_period["R_Score"] = pd.qcut(
        legacy_period["Recency"].rank(method="first"),
        5,
        labels=[5, 4, 3, 2, 1]
    ).astype(int)
    legacy_period["F_Score"] = pd.qcut(
        legacy_period["Frequency"].rank(method="first"),
        5,
        labels=[1, 2, 3, 4, 5]
    ).astype(int)

In [27]:
for tablo in (rfm1, rfm2, legacy_rfm1, legacy_rfm2):
    tablo["Segment"] = tablo.apply(
        lambda satir: segment_belirle(satir["R_Score"], satir["F_Score"]),
        axis=1
    )

In [28]:
print("Yeni d\u00f6nem 1 ilk sat\u0131rlar:")
print(rfm1[["Recency", "Frequency", "Monetary", "R_Score", "F_Score", "M_Score", "Segment"]].head())
print("Yeni d\u00f6nem 2 ilk sat\u0131rlar:")
print(rfm2[["Recency", "Frequency", "Monetary", "R_Score", "F_Score", "M_Score", "Segment"]].head())

Yeni dönem 1 ilk satırlar:
            Recency  Frequency  Monetary  R_Score  F_Score  M_Score  \
CustomerID                                                            
12347.0          54          3   1823.43        3        3        5   
12348.0          56          3   1487.24        3        3        5   
12350.0         118          1    334.40        1        1        2   
12352.0          70          5    601.18        2        5        3   
12353.0          12          1     89.00        5        1        1   

                                          Segment  
CustomerID                                         
12347.0     Dikkat Gerektirenler (Need Attention)  
12348.0     Dikkat Gerektirenler (Need Attention)  
12350.0                   Uyuyanlar (Hibernating)  
12352.0               Kaybedilemezler (Cant Lose)  
12353.0           Yeni Müşteriler (New Customers)  
Yeni dönem 2 ilk satırlar:
            Recency  Frequency  Monetary  R_Score  F_Score  M_Score  \
CustomerID   

In [29]:
# Fonksiyon cell 13'te, ilk segment atamas\u0131ndan \u00f6nce tan\u0131mland\u0131.
assert all(tablo["Segment"].ne("Tan\u0131ms\u0131z").all()
           for tablo in (rfm1, rfm2, legacy_rfm1, legacy_rfm2))
print("T\u00fcm ana/d\u00f6nem segment skorlar\u0131 tan\u0131ml\u0131 segmentlere e\u015fle\u015fti.")

Tüm ana/dönem segment skorları tanımlı segmentlere eşleşti.


In [30]:
print(segment_belirle(1, 3))
print(segment_belirle(3, 4))
print(segment_belirle(5, 5))

Risk Altındakiler (At Risk)
Sadık Müşteriler (Loyal Customers)
Şampiyonlar (Champions)


In [31]:
for tablo in (rfm1, rfm2):
    tablo["Segment"] = tablo.apply(
        lambda satir: segment_belirle(satir["R_Score"], satir["F_Score"]),
        axis=1
    )
assert rfm1["Segment"].ne("Tan\u0131ms\u0131z").all()
assert rfm2["Segment"].ne("Tan\u0131ms\u0131z").all()

In [32]:
print("TEST BAŞLADI")

print(segment_belirle(1, 3))
print(segment_belirle(3, 4))
print(segment_belirle(5, 5))

print("TEST BİTTİ")

TEST BAŞLADI
Risk Altındakiler (At Risk)
Sadık Müşteriler (Loyal Customers)
Şampiyonlar (Champions)
TEST BİTTİ


In [33]:
for tablo in (rfm1, rfm2):
    tablo["Segment"] = tablo.apply(
        lambda satir: segment_belirle(satir["R_Score"], satir["F_Score"]),
        axis=1
    )
assert rfm1["Segment"].ne("Tan\u0131ms\u0131z").all()
assert rfm2["Segment"].ne("Tan\u0131ms\u0131z").all()

In [34]:
def donem_segment_karsilastir(eski, yeni, ad):
    def ozet(tablo):
        sonuc = tablo.groupby("Segment").agg(
            Musteri_Sayisi=("Monetary", "size"),
            Toplam_Ciro=("Monetary", "sum")
        )
        sonuc["Ciro_Payi_%"] = sonuc["Toplam_Ciro"] / tablo["Monetary"].sum() * 100
        return sonuc

    eski_ozet = ozet(eski).rename(columns={
        "Musteri_Sayisi": "Musteri_Eski",
        "Toplam_Ciro": "Ciro_Eski",
        "Ciro_Payi_%": "Ciro_Payi_Eski_%"
    })
    yeni_ozet = ozet(yeni).rename(columns={
        "Musteri_Sayisi": "Musteri_Yeni",
        "Toplam_Ciro": "Ciro_Yeni",
        "Ciro_Payi_%": "Ciro_Payi_Yeni_%"
    })
    karsilastirma = eski_ozet.join(yeni_ozet, how="outer").fillna(0)
    karsilastirma["Musteri_Fark"] = karsilastirma["Musteri_Yeni"] - karsilastirma["Musteri_Eski"]
    karsilastirma["Ciro_Payi_Farki_pp"] = (
        karsilastirma["Ciro_Payi_Yeni_%"] - karsilastirma["Ciro_Payi_Eski_%"]
    )
    print(f"{ad} - old/new eligible customers: {len(eski)} / {len(yeni)}")
    print(karsilastirma.sort_values("Ciro_Payi_Yeni_%", ascending=False).round(2))

donem_segment_karsilastir(legacy_rfm1, rfm1, "Period 1")
donem_segment_karsilastir(legacy_rfm2, rfm2, "Period 2")

Period 1 - old/new eligible customers: 2767 / 2708
                                           Musteri_Eski   Ciro_Eski  \
Segment                                                               
Şampiyonlar (Champions)                             391  1297364.81   
Sadık Müşteriler (Loyal Customers)                  539   998199.09   
Uyuyanlar (Hibernating)                             699   216978.98   
Potansiyel Sadıklar (Potential Loyalists)           280   156239.70   
Uyumak Üzere Olanlar (About To Sleep)               195    64027.72   
Risk Altındakiler (At Risk)                         366   199068.06   
Dikkat Gerektirenler (Need Attention)               141    85853.05   
Umut Verenler (Promising)                            62    22633.32   
Yeni Müşteriler (New Customers)                      52    22165.42   
Kaybedilemezler (Cant Lose)                          42    57957.94   

                                           Ciro_Payi_Eski_%  Musteri_Yeni  \
Segment            

In [35]:
# Ortak puan s\u0131n\u0131rlar\u0131 iki d\u00f6nem aras\u0131nda kar\u015f\u0131la\u015ft\u0131r\u0131labilirlik sa\u011flar.
ortak_skorlar = {
    "Recency": [5, 4, 3, 2, 1],
    "Frequency": [1, 2, 3, 4, 5],
    "Monetary": [1, 2, 3, 4, 5]
}
n = len(rfm1)
for metrik, etiketler in ortak_skorlar.items():
    tum_degerler = pd.concat(
        [rfm1[metrik], rfm2[metrik]], ignore_index=True
    )
    if metrik == "Frequency":
        puanlar = frequency_score(tum_degerler)
    else:
        puanlar = pd.qcut(
            tum_degerler.rank(method="first"), 5, labels=etiketler
        ).astype(int)
    puan_kolonu = f"{metrik[0]}_Score"
    rfm1[puan_kolonu] = puanlar.iloc[:n].to_numpy()
    rfm2[puan_kolonu] = puanlar.iloc[n:].to_numpy()

for tablo in (rfm1, rfm2):
    tablo["RFM_Code"] = (
        tablo["R_Score"].astype(str)
        + tablo["F_Score"].astype(str)
        + tablo["M_Score"].astype(str)
    )
for period_table in (rfm1, rfm2):
    assert period_table.groupby("Frequency")["F_Score"].nunique().max() == 1
    assert set(period_table["F_Score"].unique()) == set(range(1, 6))
print("Period F scoring also preserves tied Frequency values and all five score levels.")

Period F scoring also preserves tied Frequency values and all five score levels.


In [36]:
for tablo in (rfm1, rfm2):
    tablo["Segment"] = tablo.apply(
        lambda satir: segment_belirle(satir["R_Score"], satir["F_Score"]),
        axis=1
    )
assert rfm1["Segment"].ne("Tan\u0131ms\u0131z").all()
assert rfm2["Segment"].ne("Tan\u0131ms\u0131z").all()

In [37]:
gecis = rfm1[["Segment"]].rename(
    columns={"Segment": "Onceki_Segment"}
).join(
    rfm2[["Segment"]].rename(columns={"Segment": "Yeni_Segment"}),
    how="outer"
)

# D\u00f6nem RFM havuzunda olmama, kesin olarak al\u0131\u015fveri\u015f yapmad\u0131 anlam\u0131na gelmez.
gecis["Onceki_Segment"] = gecis["Onceki_Segment"].fillna(
    "\u00d6nceki D\u00f6nem RFM'inde Yok"
)
gecis["Yeni_Segment"] = gecis["Yeni_Segment"].fillna(
    "\u0130kinci D\u00f6nem RFM'inde Yok"
)

gecis_tablosu = pd.crosstab(
    gecis["Onceki_Segment"], gecis["Yeni_Segment"]
)
print(gecis_tablosu)

Yeni_Segment                               Dikkat Gerektirenler (Need Attention)  \
Onceki_Segment                                                                     
Dikkat Gerektirenler (Need Attention)                                          2   
Kaybedilemezler (Cant Lose)                                                    1   
Potansiyel Sadıklar (Potential Loyalists)                                     16   
Risk Altındakiler (At Risk)                                                    2   
Sadık Müşteriler (Loyal Customers)                                             2   
Umut Verenler (Promising)                                                      4   
Uyumak Üzere Olanlar (About To Sleep)                                         12   
Uyuyanlar (Hibernating)                                                       23   
Yeni Müşteriler (New Customers)                                                6   
Önceki Dönem RFM'inde Yok                                                   

In [38]:
gecis_yuzdesi = (
    gecis_tablosu.div(gecis_tablosu.sum(axis=1), axis=0) * 100
).round(2)

print("Sat\u0131r baz\u0131nda d\u00f6nemler aras\u0131 ge\u00e7i\u015f oranlar\u0131 (%):")
print(gecis_yuzdesi)

Satır bazında dönemler arası geçiş oranları (%):
Yeni_Segment                               Dikkat Gerektirenler (Need Attention)  \
Onceki_Segment                                                                     
Dikkat Gerektirenler (Need Attention)                                       3.77   
Kaybedilemezler (Cant Lose)                                                 6.25   
Potansiyel Sadıklar (Potential Loyalists)                                   3.35   
Risk Altındakiler (At Risk)                                                 2.94   
Sadık Müşteriler (Loyal Customers)                                          1.65   
Umut Verenler (Promising)                                                   3.70   
Uyumak Üzere Olanlar (About To Sleep)                                       3.72   
Uyuyanlar (Hibernating)                                                     2.42   
Yeni Müşteriler (New Customers)                                             2.86   
Önceki Dönem RFM'inde Yok  